In [ ]:
import argparse
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from datetime import datetime
import time
import os
import sys

# 📌 Argumentos de Línea de Comandos
parser = argparse.ArgumentParser(description="Spider para extraer PDFs del BORME por fecha o archivo de fechas.")
parser.add_argument('--file', type=str, help='Archivo de texto con fechas en formato YYYY-MM-DD, una por línea.')
parser.add_argument('--date', type=str, help='Una fecha específica en formato YYYY-MM-DD.')

args = parser.parse_args()

# 📁 Crear Carpeta para Guardar las URLs por Fecha
output_folder = "urls_por_fecha"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"✅ Carpeta '{output_folder}' creada correctamente.")
else:
    print(f"📂 Carpeta '{output_folder}' ya existe.")

# 🌐 Inicialización del Navegador
driver = webdriver.Chrome()
driver.maximize_window()
url = "https://www.boe.es/diario_borme/"
driver.get(url)

# 📅 Función para Procesar una Fecha Específica
def procesar_fecha(fecha_argumento):
    try:
        print(f"\n📅 Procesando fecha: {fecha_argumento}")
        fecha = datetime.strptime(fecha_argumento, "%Y-%m-%d")
        fecha_formateada = fecha.strftime("%d/%m/%Y")
        
        # **Seleccionar Fecha en el Calendario**
        time.sleep(2)
        input_calendar = driver.find_element(By.ID, "fechaBORME")
        input_calendar.clear()
        input_calendar.send_keys(fecha_formateada)
        input_calendar.send_keys(Keys.RETURN)
        time.sleep(3)
        print(f"✅ Fecha {fecha_formateada} seleccionada correctamente.")
        
        # **Comprobar si la fecha tiene boletín**
        try:
            error_msg = driver.find_element(By.XPATH, "//p[contains(@class, 'caja gris error') and contains(text(), 'No se ha publicado boletín')]")
            print(f"⚠️ No hay boletín publicado para la fecha {fecha_argumento}.")
            driver.back()
            time.sleep(3)
            print("🔄 Usando 'driver.back()' para regresar a la página principal.")
            return
        
        except Exception:
            print("✅ Hay boletín publicado. Continuando con el procesamiento...")
        
        # **Interactuar con el Dropdown Personalizado**
        dropdown_button = driver.find_element(By.CSS_SELECTOR, 'label[for="dropDownSec"]')
        dropdown_button.click()
        time.sleep(1)
        opcion_actos_inscritos = driver.find_element(By.XPATH, "//a[contains(text(), 'Actos inscritos')]")
        opcion_actos_inscritos.click()
        time.sleep(3)
        print("✅ Opción 'Actos inscritos' seleccionada correctamente.")

        # **Extraer URLs de PDFs**
        pdf_links = driver.find_elements(By.TAG_NAME, "a")
        pdf_urls = [
            link.get_attribute('href') for link in pdf_links
            if link.get_attribute('href') and link.get_attribute('href').endswith('.pdf')
        ]

        if pdf_urls:
            print(f"✅ Se encontraron {len(pdf_urls)} enlaces a PDFs.")
            
            # Guardar las URLs en un archivo
            output_file = os.path.join(output_folder, f"{fecha_argumento}.txt")
            with open(output_file, "w") as file:
                for url in pdf_urls:
                    file.write(url + "\n")
            print(f"✅ URLs guardadas en '{output_file}'.")
            
            # Volver a la página principal de BORME
            boton_borme = driver.find_element(By.XPATH, '//a[@href="/diario_borme/"]')
            boton_borme.click()
            print("🔄 Volviendo a la página principal de BORME...")
            time.sleep(3)
        else:
            print("⚠️ No se encontraron enlaces a PDFs. Usando 'driver.back()' para regresar.")
            driver.back()
            time.sleep(3)

    except Exception as e:
        print(f"❌ Error al procesar la fecha {fecha_argumento}: {e}")


# 🚀 Procesamiento Principal
try:
    if args.file:
        # Procesar desde un archivo
        if not os.path.exists(args.file):
            print(f"❌ El archivo '{args.file}' no existe.")
            sys.exit(1)
        
        with open(args.file, "r") as file:
            fechas = file.readlines()
            fechas = [fecha.strip() for fecha in fechas if fecha.strip()]
        
        print(f"✅ Se han encontrado {len(fechas)} fechas para procesar en el archivo.")
        for fecha in fechas:
            procesar_fecha(fecha)
    
    elif args.date:
        # Procesar una fecha específica
        try:
            datetime.strptime(args.date, "%Y-%m-%d")  # Validar formato de fecha
            procesar_fecha(args.date)
        except ValueError:
            print("❌ La fecha proporcionada no tiene el formato correcto (YYYY-MM-DD).")
            sys.exit(1)
    
    else:
        print("❌ Debes proporcionar un archivo (--file) o una fecha (--date). Usa --help para más información.")
        sys.exit(1)

except Exception as e:
    print(f"❌ Error inesperado: {e}")

finally:
    driver.quit()
    print("🛑 Navegador cerrado correctamente.")